# Creating Redshift Cluster using the AWS python SDK 
## Infrastructure-as-code

In [1]:
import pandas as pd
import boto3
import json

# STEP 0: Make sure you have an AWS secret and access key

- Create a new IAM user in your AWS account
- Give it `AdministratorAccess`, From `Attach existing policies directly` Tab
- Take note of the access key and secret 
- Edit the file `dwh.cfg` in the same folder as this notebook and fill
<font color='red'>
<BR>
[AWS]<BR>
KEY= YOUR_AWS_KEY<BR>
SECRET= YOUR_AWS_SECRET<BR>
<font/>

# Load DWH Params from a file

In [2]:
import getpass
import configparser
config = configparser.ConfigParser()
config.read_file(open('dwh.cfg'))

if not config.get('AWS','KEY'):
    KEY                = getpass.getpass(prompt='KEY:')
else:
    KEY                = config.get('AWS','KEY')
if not config.get('AWS','SECRET'):
    SECRET             = getpass.getpass(prompt='SECRET:')
else:
    SECRET             = config.get('AWS','SECRET')

if not config.get('IAM_ROLE','ARN'):
    ARN                = input(prompt='ARN:')
else:
    ARN                = config.get('IAM_ROLE','ARN')

DWH_IAM_ROLE_NAME      = config.get('DWH','DWH_IAM_ROLE_NAME')
DWH_CLUSTER_TYPE       = config.get('DWH','DWH_CLUSTER_TYPE')
DWH_NUM_NODES          = config.get('DWH','DWH_NUM_NODES')
DWH_NODE_TYPE          = config.get('DWH','DWH_NODE_TYPE')
DWH_CLUSTER_IDENTIFIER = config.get('DWH','DWH_CLUSTER_IDENTIFIER')

if not config.get('CLUSTER','HOST'):
    HOST               = input(prompt='HOST:')
else:
    HOST               = config.get('CLUSTER','HOST')
DB_NAME                = config.get('CLUSTER','DB_NAME')
DB_USER                = config.get('CLUSTER','DB_USER')
if not config.get('CLUSTER','DB_PASSWORD'):
    DB_PASSWORD        = getpass.getpass(prompt='DB_PASSWORD:')
else:
    DB_PASSWORD        = config.get('CLUSTER','DB_PASSWORD') 
DB_PORT                = config.get('CLUSTER','DB_PORT')

LOG_DATA               = config.get('S3','LOG_DATA')
LOG_JSONPATH           = config.get('S3','LOG_JSONPATH')
SONG_DATA              = config.get('S3','SONG_DATA')
REGION                 = config.get('S3','REGION')

df = pd.DataFrame({'Param':
                  [ 'KEY','SECRET' \
                   ,'ARN' \
                   ,'DWH_IAM_ROLE_NAME','DWH_CLUSTER_TYPE','DWH_NUM_NODES','DWH_NODE_TYPE','DWH_CLUSTER_IDENTIFIER' \
                   ,'HOST','DB_NAME','DB_USER','DB_PASSWORD','DB_PORT' \
                   ,'LOG_DATA','LOG_JSONPATH','SONG_DATA','REGION'] \
                  ,'Value': \
                  [ KEY,SECRET \
                   ,ARN \
                   ,DWH_IAM_ROLE_NAME,DWH_CLUSTER_TYPE,DWH_NUM_NODES,DWH_NODE_TYPE,DWH_CLUSTER_IDENTIFIER \
                   ,HOST,DB_NAME,DB_USER,DB_PASSWORD,DB_PORT \
                   ,LOG_DATA,LOG_JSONPATH,SONG_DATA,REGION \
                  ] \
                  })
df.where((df != KEY) & (df != SECRET) & (df != DB_PASSWORD),'********')

KEY:········
SECRET:········
ARN:
HOST:
DB_PASSWORD:········


,Param,Value
0,KEY,********
1,SECRET,********
2,ARN,
3,DWH_IAM_ROLE_NAME,dwhRole
4,DWH_CLUSTER_TYPE,multi-node
5,DWH_NUM_NODES,4
6,DWH_NODE_TYPE,dc2.large
7,DWH_CLUSTER_IDENTIFIER,dwhCluster
8,HOST,
9,DB_NAME,dwh


# Create clients for IAM, EC2, S3 and Redshift

In [3]:
ec2 = boto3.resource('ec2',
                       region_name=REGION,
                       aws_access_key_id=KEY,
                       aws_secret_access_key=SECRET
                    )

s3 = boto3.resource('s3',
                       region_name=REGION,
                       aws_access_key_id=KEY,
                       aws_secret_access_key=SECRET
                   )

iam = boto3.client('iam',aws_access_key_id=KEY,
                     aws_secret_access_key=SECRET,
                     region_name=REGION
                  )

redshift = boto3.client('redshift',
                       region_name=REGION,
                       aws_access_key_id=KEY,
                       aws_secret_access_key=SECRET
                       )

# Test credentials by viewing sample data sources on S3

In [4]:
try:
    sampleDbBucket =  s3.Bucket("awssampledbuswest2")
    for obj in sampleDbBucket.objects.filter(Prefix="ssbgz"):
        print(obj)
except Exception as e:
    print(e)

s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/customer0002_part_00.gz')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/dwdate.tbl.gz')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/lineorder0000_part_00.gz')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/lineorder0001_part_00.gz')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/lineorder0002_part_00.gz')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/lineorder0003_part_00.gz')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/lineorder0004_part_00.gz')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/lineorder0005_part_00.gz')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/lineorder0006_part_00.gz')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='ssbgz/lineorder0007_part_00.gz')
s3.ObjectSummary(bucket_name='awssampledbuswest2', key='s

# STEP 1: IAM ROLE
- Create an IAM Role that makes Redshift able to access S3 bucket (ReadOnly)

In [5]:
from botocore.exceptions import ClientError

#1.1 Create the role, 
try:
    print("1.1 Creating a new IAM Role") 
    dwhRole = iam.create_role(
        Path='/',
        RoleName=DWH_IAM_ROLE_NAME,
        Description = "Allows Redshift clusters to call AWS services on your behalf.",
        AssumeRolePolicyDocument=json.dumps(
            {'Statement': [{'Action': 'sts:AssumeRole',
               'Effect': 'Allow',
               'Principal': {'Service': 'redshift.amazonaws.com'}}],
             'Version': '2012-10-17'})
    )    
except Exception as e:
    print(e)
    
    
print("1.2 Attaching Policy")

iam.attach_role_policy(RoleName=DWH_IAM_ROLE_NAME,
                       PolicyArn="arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess"
                      )['ResponseMetadata']['HTTPStatusCode']

print("1.3 Get the IAM role ARN")
roleArn = iam.get_role(RoleName=DWH_IAM_ROLE_NAME)['Role']['Arn']

print(roleArn)

1.1 Creating a new IAM Role
1.2 Attaching Policy
1.3 Get the IAM role ARN
arn:aws:iam::840977349409:role/dwhRole


# STEP 2:  Redshift Cluster

- Create a RedShift Cluster
- For complete arguments to `create_cluster`, see [docs](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/redshift.html#Redshift.Client.create_cluster)

In [6]:
try:
    response = redshift.create_cluster(        
        #HW
        ClusterType=DWH_CLUSTER_TYPE,
        NodeType=DWH_NODE_TYPE,
        NumberOfNodes=int(DWH_NUM_NODES),

        #Identifiers & Credentials
        DBName=DB_NAME,
        ClusterIdentifier=DWH_CLUSTER_IDENTIFIER,
        MasterUsername=DB_USER,
        MasterUserPassword=DB_PASSWORD,
        
        #Roles (for s3 access)
        IamRoles=[roleArn]  
    )
except Exception as e:
    print(e)

## 2.1 *Describe* the cluster to see its status
- run this block several times until the cluster status becomes `Available`

In [7]:
def prettyRedshiftProps(props):
    pd.set_option('display.max_colwidth', -1)
    keysToShow = ["ClusterIdentifier", "NodeType", "ClusterStatus", "MasterUsername", "DBName", "Endpoint", "NumberOfNodes", 'VpcId']
    x = [(k, v) for k,v in props.items() if k in keysToShow]
    return pd.DataFrame(data=x, columns=["Key", "Value"])

myClusterProps = redshift.describe_clusters(ClusterIdentifier=DWH_CLUSTER_IDENTIFIER)['Clusters'][0]
prettyRedshiftProps(myClusterProps)

,Key,Value
0,ClusterIdentifier,dwhcluster
1,NodeType,dc2.large
2,ClusterStatus,creating
3,MasterUsername,dwhuser
4,DBName,dwh
5,VpcId,vpc-7fc8ea07
6,NumberOfNodes,4


In [8]:
import time
start = time.perf_counter()
t = 0
df2 = prettyRedshiftProps(myClusterProps)

try:
    while df2.where(df2.isin(['available']) != True).isnull().sum().sum() == 0:
        time.sleep(10)
        t += 10
        myClusterProps = redshift.describe_clusters(ClusterIdentifier=DWH_CLUSTER_IDENTIFIER)['Clusters'][0]
        df2 = prettyRedshiftProps(myClusterProps)
        if t <= 600:
            continue
        else:
            print('timeout')
            break
except Exception as e:
    print(e)
finally:
    for i in df2:
        for x in df2[i]:
            print(i,x)
            
end = time.perf_counter()
execution_time = end - start
print('Execution time: {} seconds'.format(execution_time))

<built-in method view of numpy.ndarray object at 0x7f9b4e2d8090> returned a result with an error set
Key ClusterIdentifier
Key NodeType
Key ClusterStatus
Key MasterUsername
Key DBName
Key Endpoint
Key VpcId
Key NumberOfNodes
Value dwhcluster
Value dc2.large
Value available
Value dwhuser
Value dwh
Value {'Address': 'dwhcluster.c57vg86cbbes.us-west-2.redshift.amazonaws.com', 'Port': 5439}
Value vpc-7fc8ea07
Value 4
Execution time: 175.963501432 seconds


<h2> 2.2 Take note of the cluster <font color='red'> endpoint and role ARN </font> </h2>

<font color='red'>DO NOT RUN THIS unless the cluster status becomes "Available" </font>

In [9]:
DWH_ENDPOINT = myClusterProps['Endpoint']['Address']
DWH_ROLE_ARN = myClusterProps['IamRoles'][0]['IamRoleArn']

if not HOST:
    HOST = DWH_ENDPOINT
    
if not ARN:
    ARN = DWH_ROLE_ARN

print("DWH_ENDPOINT :: ", DWH_ENDPOINT)
print("DWH_ROLE_ARN :: ", DWH_ROLE_ARN)

DWH_ENDPOINT ::  dwhcluster.c57vg86cbbes.us-west-2.redshift.amazonaws.com
DWH_ROLE_ARN ::  arn:aws:iam::840977349409:role/dwhRole


## STEP 3: Open an incoming  TCP port to access the cluster endpoint

In [10]:
try:
    vpc = ec2.Vpc(id=myClusterProps['VpcId'])
    defaultSg = list(vpc.security_groups.all())[0]
    print(defaultSg)
    defaultSg.authorize_ingress(
        GroupName=defaultSg.group_name,
        CidrIp='0.0.0.0/0',
        IpProtocol='TCP',
        FromPort=int(DB_PORT),
        ToPort=int(DB_PORT)
    )
except Exception as e:
    print(e)

ec2.SecurityGroup(id='sg-869d31bb')


# STEP 4: Make sure you can connect to the cluster

In [11]:
#%load_ext sql

#conn_string="postgresql://{}:{}@{}:{}/{}".format(DB_USER, DB_PASSWORD, HOST, DB_PORT, DB_NAME)
#print(conn_string)
#%sql $conn_string

#%sql SELECT current_timestamp

import psycopg2

try:
    conn = psycopg2.connect(dbname=DB_NAME,host=HOST,port=DB_PORT,user=DB_USER,password=DB_PASSWORD)
except psycopg2.Error as e: 
    print(e)

In [12]:
try: 
    cur = conn.cursor()
except psycopg2.Error as e: 
    print(e)

In [13]:
conn.set_session(autocommit=True)

In [14]:
try:
    cur.execute('select version()')
except psycopg2.Error as e:
    print(e)
    
cur.fetchone()

('PostgreSQL 8.0.2 on i686-pc-linux-gnu, compiled by GCC gcc (GCC) 3.4.2 20041017 (Red Hat 3.4.2-6.fc3), Redshift 1.0.25599',)

# Build ETL Pipeline

In [15]:
#!python3 create_tables.py
#%run -i 'create_tables.py'

import create_tables
cluster = [HOST,DB_NAME,DB_USER,DB_PASSWORD,DB_PORT]
create_tables.main(cluster)

In [16]:
table_columns = {'staging_songs':[],'staging_events':[],'users':[],'artists':[],'songs':[],'time':[],'songplays':[]}

for tbl in table_columns:
    sql = 'select * from {} limit 0'.format(tbl)
    try: 
        cur.execute(sql)
    except psycopg2.Error as e:
        print(e)
    for desc in cur.description:
        table_columns[tbl].append(desc[0])

table_columns

{'staging_songs': ['num_songs',
  'artist_id',
  'artist_latitude',
  'artist_longitude',
  'artist_location',
  'artist_name',
  'song_id',
  'title',
  'duration',
  'year'],
 'staging_events': ['artist',
  'auth',
  'firstname',
  'gender',
  'iteminsession',
  'lastname',
  'length',
  'level',
  'location',
  'method',
  'page',
  'registration',
  'sessionid',
  'song',
  'status',
  'ts',
  'useragent',
  'user_id'],
 'users': ['user_id', 'first_name', 'last_name', 'gender', 'level'],
 'artists': ['artist_id', 'name', 'location', 'latitude', 'longitude'],
 'songs': ['song_id', 'title', 'artist_id', 'year', 'duration'],
 'time': ['start_time', 'hour', 'day', 'week', 'month', 'year', 'weekday'],
 'songplays': ['songplay_id',
  'start_time',
  'user_id',
  'level',
  'song_id',
  'artist_id',
  'session_id',
  'location',
  'user_agent']}

In [17]:
start = time.perf_counter()

#!python3 etl.py
#%run -i 'etl.py'

import etl
cluster = [HOST,DB_NAME,DB_USER,DB_PASSWORD,DB_PORT]
etl.main(cluster,ARN)

end = time.perf_counter()
execution_time = end - start
print('Execution time: {} seconds'.format(execution_time))

Execution time: 290.45153755499996 seconds


In [18]:
table_counts = {'staging_songs':[],'staging_events':[],'users':[],'artists':[],'songs':[],'time':[],'songplays':[]}

for tbl in table_counts:
    sql = 'select count(*) from {} limit 1'.format(tbl)
    try: 
        cur.execute(sql)
    except psycopg2.Error as e:
        print(e)
    table_counts[tbl].append(cur.fetchone()[0])

table_counts

{'staging_songs': [14896],
 'staging_events': [8056],
 'users': [97],
 'artists': [9553],
 'songs': [14896],
 'time': [8023],
 'songplays': [326]}

In [19]:
try: 
    cur.execute('select "schema", "table", diststyle \
                 from SVV_TABLE_INFO; \
                ')
except psycopg2.Error as e:
    print(e)
    
cur.fetchmany(10)

[('public', 'users', 'ALL'),
 ('public', 'songs', 'KEY(song_id)'),
 ('public', 'staging_songs', 'AUTO(EVEN)'),
 ('public', 'songplays', 'AUTO(EVEN)'),
 ('public', 'time', 'AUTO(EVEN)'),
 ('public', 'artists', 'AUTO(EVEN)'),
 ('public', 'staging_events', 'AUTO(ALL)')]

# STEP 5: Clean up your resources

<b><font color='red'>DO NOT RUN THIS UNLESS YOU ARE SURE <br/> 
    </span></b>

In [20]:
#### CAREFUL!!
#-- Uncomment & run to delete the created resources
redshift.delete_cluster( ClusterIdentifier=DWH_CLUSTER_IDENTIFIER,  SkipFinalClusterSnapshot=True)
#### CAREFUL!!

{'Cluster': {'ClusterIdentifier': 'dwhcluster',
  'NodeType': 'dc2.large',
  'ClusterStatus': 'deleting',
  'ClusterAvailabilityStatus': 'Modifying',
  'MasterUsername': 'dwhuser',
  'DBName': 'dwh',
  'Endpoint': {'Address': 'dwhcluster.c57vg86cbbes.us-west-2.redshift.amazonaws.com',
   'Port': 5439},
  'ClusterCreateTime': datetime.datetime(2021, 4, 20, 1, 1, 55, 745000, tzinfo=tzutc()),
  'AutomatedSnapshotRetentionPeriod': 1,
  'ManualSnapshotRetentionPeriod': -1,
  'ClusterSecurityGroups': [],
  'VpcSecurityGroups': [{'VpcSecurityGroupId': 'sg-869d31bb',
    'Status': 'active'}],
  'ClusterParameterGroups': [{'ParameterGroupName': 'default.redshift-1.0',
    'ParameterApplyStatus': 'in-sync'}],
  'ClusterSubnetGroupName': 'default',
  'VpcId': 'vpc-7fc8ea07',
  'AvailabilityZone': 'us-west-2b',
  'PreferredMaintenanceWindow': 'thu:13:30-thu:14:00',
  'PendingModifiedValues': {},
  'ClusterVersion': '1.0',
  'AllowVersionUpgrade': True,
  'NumberOfNodes': 4,
  'PubliclyAccessible':

- run this block several times until the cluster really deleted

In [21]:
myClusterProps = redshift.describe_clusters(ClusterIdentifier=DWH_CLUSTER_IDENTIFIER)['Clusters'][0]
prettyRedshiftProps(myClusterProps)

,Key,Value
0,ClusterIdentifier,dwhcluster
1,NodeType,dc2.large
2,ClusterStatus,deleting
3,MasterUsername,dwhuser
4,DBName,dwh
5,Endpoint,"{'Address': 'dwhcluster.c57vg86cbbes.us-west-2.redshift.amazonaws.com', 'Port': 5439}"
6,VpcId,vpc-7fc8ea07
7,NumberOfNodes,4


In [22]:
start = time.perf_counter()
t2 = 1
df3 = prettyRedshiftProps(myClusterProps)

try:
    while df3[['Value'][0]][2] == 'deleting':
        time.sleep(10)
        t2 += 10
        myClusterProps = redshift.describe_clusters(ClusterIdentifier=DWH_CLUSTER_IDENTIFIER)['Clusters'][0]
        df3 = prettyRedshiftProps(myClusterProps)
        if t2 <= 600:
            continue
        else:
            print('timeout')
            break
except Exception as e:
    print(e)
finally:
    for i in df3:
        for x in df3[i]:
            print(i,x)
            
end = time.perf_counter()
execution_time = end - start
print('Execution time: {} seconds'.format(execution_time))

An error occurred (ClusterNotFound) when calling the DescribeClusters operation: Cluster dwhcluster not found.
Key ClusterIdentifier
Key NodeType
Key ClusterStatus
Key MasterUsername
Key DBName
Key Endpoint
Key VpcId
Key NumberOfNodes
Value dwhcluster
Value dc2.large
Value deleting
Value dwhuser
Value dwh
Value {'Port': 5439}
Value vpc-7fc8ea07
Value 4
Execution time: 164.78698879599995 seconds


In [23]:
try:
    vpc = ec2.Vpc(id=myClusterProps['VpcId'])
    defaultSg = list(vpc.security_groups.all())[0]
    print(defaultSg)
    defaultSg.revoke_ingress(
        GroupName=defaultSg.group_name,
        CidrIp='0.0.0.0/0',
        IpProtocol='TCP',
        FromPort=int(DB_PORT),
        ToPort=int(DB_PORT)
    )
except Exception as e:
    print(e)

ec2.SecurityGroup(id='sg-869d31bb')


In [24]:
#### CAREFUL!!
#-- Uncomment & run to delete the created resources
iam.detach_role_policy(RoleName=DWH_IAM_ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess")
iam.delete_role(RoleName=DWH_IAM_ROLE_NAME)
#### CAREFUL!!

{'ResponseMetadata': {'RequestId': '978cbfc5-26fc-413a-a306-7e9805626087',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '978cbfc5-26fc-413a-a306-7e9805626087',
   'content-type': 'text/xml',
   'content-length': '200',
   'date': 'Tue, 20 Apr 2021 01:09:51 GMT'},
  'RetryAttempts': 0}}